# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd 
import numpy as np 

In [2]:

import os
os.listdir()


['rappiplus_marketing_spend.csv',
 'rappiplus_catalog.csv',
 '.ipynb_checkpoints',
 'marketing_clean.csv',
 'rappiplus_orders_raw.csv',
 'S12 Estudiante_Proyecto_Final.ipynb',
 'orders_clean.csv',
 'catalog_clean.csv']

In [3]:
# cargar archivos
orders =pd.read_csv('rappiplus_orders_raw.csv')
catalog = pd.read_csv('rappiplus_catalog.csv')
marketing = pd.read_csv('rappiplus_marketing_spend.csv')

In [4]:
# explorar datasets
#orders
print("===== ORDERS =====")
print(orders.head())

print("information:")

print(orders.info())
print("/Estadisticas:")
print(orders.describe(include='all'))

===== ORDERS =====
  id_pedido id_usuario fecha_hora_pedido       pais dispositivo  \
0   order_0  user_6993        2025-05-22  Argentina     desktop   
1   order_1  user_1329        2025-06-15     Mexico     desktop   
2   order_2  user_3194        2025-05-02  Argentina     desktop   
3   order_3  user_4510        2025-06-09   Colombia      mobile   
4   order_4  user_5044        2025-03-30  Argentina     desktop   

  fuente_referencia       nombre_producto categoria_producto  cantidad  \
0           organic       Jacket-Winter-M               Moda       2.0   
1       paid_search  Tablet-Standard-64GB        Electronica       1.0   
2            social        Blender-XL-Red              Hogar       2.0   
3            social  Tablet-Standard-64GB        Electronica       1.0   
4       paid_search        Blender-XL-Red              Hogar       1.0   

   precio_unitario  monto_descuento  monto_total  
0           332.69              0.0       665.37  
1           176.86             

In [5]:
orders['id_pedido'].duplicated().sum()

100

In [6]:

#catalog
print("===== CATALOG =====")
print(catalog.head())
print("information:")
print(catalog.info())
print("/Estadisticas:")
print(catalog.describe(include='all'))


===== CATALOG =====
        nombre_producto categoria_producto  costo_unitario  \
0    Laptop-Gaming-16GB        Electrónica          280.68   
1       Phone-Pro-128GB        Electrónica           10.12   
2  Tablet-Standard-64GB        Electrónica           25.21   
3        Blender-XL-Red              Hogar          176.64   
4      Vacuum-Pro-Black              Hogar           16.60   

                 proveedor  
0   Fuller, Pena and Myers  
1                 King Ltd  
2               Bowers LLC  
3                Long-Reid  
4  Rivera, Carr and Finley  
information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object

In [7]:
#Marketing
print("===== MARKETING =====")
print(marketing.head())
print("information:")
print(marketing.info())
print("/Estadisticas:")
print(marketing.describe(include='all'))


===== MARKETING =====
        fecha      pais            id_campaña        canal    gasto
0  2025-01-01    Mexico        organic_Mexico      organic  2446.25
1  2025-01-01    Mexico    paid_search_Mexico  paid_search  2704.34
2  2025-01-01    Mexico         social_Mexico       social  2045.01
3  2025-01-01  Colombia      organic_Colombia      organic  2597.21
4  2025-01-01  Colombia  paid_search_Colombia  paid_search  1771.40
information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB
None
/Estadisticas:
             fecha      pais        id_campaña        canal       gasto
count         1620 

---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [8]:
#Validar y convertir fechas al formato correcto
orders['fecha_hora_pedido']=pd.to_datetime(orders['fecha_hora_pedido'])
marketing['fecha']=pd.to_datetime(marketing['fecha'])

In [9]:

#Revisar variables numéricas (sin negativos o ceros inválidos)

# orders_cantidad 
# Calcular los cuartiles

Q1 = orders["cantidad"].quantile(0.25)

Q3 = orders["cantidad"].quantile(0.75)

IQR = Q3 - Q1

# Límites

limite_inferior = Q1 - 1.5 * IQR

limite_superior = Q3 + 1.5 * IQR

# Mostrar los outliers

outliers = orders[

    (orders["cantidad"] < limite_inferior) |

    (orders["cantidad"] > limite_superior)

]

print(outliers)


       id_pedido id_usuario fecha_hora_pedido       pais dispositivo  \
266    order_266  user_7011        2025-03-13        NaN     desktop   
267    order_267  user_1087        2025-05-07        NaN     desktop   
268    order_268    user_84        2025-02-19        NaN     desktop   
269    order_269  user_3323        2025-05-25        NaN     desktop   
3521  order_3521  user_5812        2025-02-03     Mexico      mobile   
3522  order_3522  user_3575        2025-03-29  Argentina     desktop   
3586  order_3586  user_3380        2025-02-03     Mexico      mobile   
3643  order_3643  user_4440        2025-01-07   Colombia     desktop   
3656  order_3656   user_884        2025-01-01  Argentina      mobile   
3668  order_3668  user_7270        2025-06-24     Mexico      mobile   
3689  order_3689  user_6566        2025-06-16     mexico     desktop   
3722  order_3722  user_4723        2025-05-09  Argentina      mobile   
3726  order_3726  user_2536        2025-02-12   Colombia     des

In [10]:
# Eliminar Valores inferiores a 0 (negativos)
orders = orders[orders["cantidad"] > 0]

In [11]:
(orders["cantidad"] < 0).sum()

0

In [12]:
# Eliminar outliers superiores a 10 
orders = orders[orders["cantidad"] <= 10]

In [13]:
(orders["cantidad"]> 10 ).sum()

0

In [14]:
#Verificar consistencia de montos
# Eliminar montos negativos
orders = orders[orders["monto_total"] > 0]

In [15]:
(orders["cantidad"] < 0 ).sum()

0

In [16]:
#Eliminar duplicados
# Eliminar id_pedido
orders = orders.drop_duplicates(subset="id_pedido", keep="first")

In [17]:
orders['id_pedido'].duplicated().sum()

0

In [18]:
#Revision Variables categoricas 

In [19]:
# Orders

categoricas_order = [

    "id_pedido",

    "id_usuario",

    "pais",

    "dispositivo",

    "fuente_referencia",

    "nombre_producto",

    "categoria_producto"

]

for col in categoricas_order:

    print(f"\n===== {col.upper()} =====")

    print("Valores únicos:", orders[col].nunique())

    print("Valores nulos:", orders[col].isna().sum())

    print(orders[col].value_counts(dropna=False))


===== ID_PEDIDO =====
Valores únicos: 24936
Valores nulos: 0
order_21175    1
order_3578     1
order_12107    1
order_16692    1
order_21708    1
              ..
order_7869     1
order_21545    1
order_24773    1
order_19356    1
order_24525    1
Name: id_pedido, Length: 24936, dtype: int64

===== ID_USUARIO =====
Valores únicos: 7640
Valores nulos: 0
user_7769    11
user_6272    10
user_5748    10
user_1554    10
user_1822    10
             ..
user_1274     1
user_1892     1
user_7064     1
user_2110     1
user_3399     1
Name: id_usuario, Length: 7640, dtype: int64

===== PAIS =====
Valores únicos: 6
Valores nulos: 296
Colombia     7465
Mexico       7461
Argentina    7236
mexico        861
colombia      822
argentina     795
NaN           296
Name: pais, dtype: int64

===== DISPOSITIVO =====
Valores únicos: 2
Valores nulos: 20
desktop    12676
mobile     12240
NaN           20
Name: dispositivo, dtype: int64

===== FUENTE_REFERENCIA =====
Valores únicos: 3
Valores nulos: 30
social

In [20]:
# Cambiar formato de los nombres de los paises
orders["pais"] = orders["pais"].str.title()

In [21]:
# Eliminar valores nulos 
orders = orders.dropna(subset=["pais"])

#Se identificaron 296 valores nulos en la variable pais, equivalentes al 1.2% del total de registros. Dado que representan una proporción reducida y que la variable es fundamental para el análisis geográfico del negocio, se optó por eliminar dichos registros para garantizar la consistencia y confiabilidad de los resultados.

In [22]:
orders = orders.dropna(subset=["dispositivo"])
orders = orders.dropna(subset=["fuente_referencia"])
orders = orders.dropna(subset=["nombre_producto"])
orders = orders.dropna(subset=["categoria_producto"])

In [23]:
#Revision Categoricas de catalog
categoricas_catalog = [

    "nombre_producto",

    "categoria_producto",

    "proveedor"

]

for col in categoricas_catalog:

    print(f"\n===== {col.upper()} =====")

    print("Valores únicos:", catalog[col].nunique())

    print("Valores nulos:", catalog[col].isna().sum())

    print(catalog[col].value_counts(dropna=False))


===== NOMBRE_PRODUCTO =====
Valores únicos: 7
Valores nulos: 0
Laptop-Gaming-16GB      1
Sneakers-Urban-42       1
Phone-Pro-128GB         1
Blender-XL-Red          1
Tablet-Standard-64GB    1
Vacuum-Pro-Black        1
Jacket-Winter-M         1
Name: nombre_producto, dtype: int64

===== CATEGORIA_PRODUCTO =====
Valores únicos: 3
Valores nulos: 0
Electrónica    3
Hogar          2
Moda           2
Name: categoria_producto, dtype: int64

===== PROVEEDOR =====
Valores únicos: 7
Valores nulos: 0
Long-Reid                  1
Greene-Smith               1
Rivera, Carr and Finley    1
Bowers LLC                 1
Fuller, Pena and Myers     1
King Ltd                   1
Mcmillan-Rhodes            1
Name: proveedor, dtype: int64


In [24]:
categoricas_marketing = [

    "pais",

    "id_campaña",

    "canal"

]

for col in categoricas_marketing:

    print(f"\n===== {col.upper()} =====")

    print("Valores únicos:", marketing[col].nunique())

    print("Valores nulos:", marketing[col].isna().sum())

    print(marketing[col].value_counts(dropna=False))


===== PAIS =====
Valores únicos: 3
Valores nulos: 0
Colombia     540
Mexico       540
Argentina    540
Name: pais, dtype: int64

===== ID_CAMPAÑA =====
Valores únicos: 9
Valores nulos: 0
social_Argentina         180
organic_Colombia         180
organic_Mexico           180
paid_search_Argentina    180
organic_Argentina        180
paid_search_Mexico       180
social_Colombia          180
social_Mexico            180
paid_search_Colombia     180
Name: id_campaña, dtype: int64

===== CANAL =====
Valores únicos: 3
Valores nulos: 101
paid_search    507
organic        506
social         506
NaN            101
Name: canal, dtype: int64


In [25]:
# Eliminar valores nulos 
marketing = marketing.dropna(subset=["canal"])

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [26]:

# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)


---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [27]:
# Unir orders y catalog
orders_cost = orders.merge(

    catalog[["nombre_producto", "costo_unitario"]],

    on="nombre_producto",

    how="left"

)

In [28]:
orders_cost.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,costo_unitario
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37,189.31
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86,25.21
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99,176.64
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87,25.21
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28,176.64


In [29]:
# ==== RENTABILIDAD DEL NEGOCIO =====

#costo por pedido
orders_cost["costo_total"] = (

    orders_cost["cantidad"] *

    orders_cost["costo_unitario"]

)


In [30]:
#Costo total empresa
costo_total_empresa = orders_cost["costo_total"].sum()

print("Costo total de la empresa:", round(costo_total_empresa, 2))

Costo total de la empresa: 3783478.8


In [31]:
#Revenue (Ingreso total)
revenue = orders_cost["monto_total"].sum()

print("Revenue:", revenue)

Revenue: 9491675.14


In [32]:
#inversion en marketing
marketing_total = marketing["gasto"].sum()

print("Marketing:", marketing_total)

Marketing: 2694664.4299999997


In [33]:
#profit
profit = revenue - costo_total_empresa - marketing_total

print("Profit:", profit)

Profit: 3013531.910000001


In [34]:
#Comportamiento de ventas

#Ticket promedio
ticket_promedio = orders_cost["monto_total"].mean()

print("Ticket promedio:", ticket_promedio)

Ticket promedio: 385.9973623424156


In [35]:
#cantidad promedio por pedido
cantidad_promedio = orders_cost["cantidad"].mean()

print("Cantidad promedio:", cantidad_promedio)

Cantidad promedio: 1.5052053680357869


In [36]:
#producto mas vendido
producto_mas_vendido = (

    orders_cost.groupby("nombre_producto")["cantidad"]

    .sum()

    .sort_values(ascending=False)

)

print(producto_mas_vendido)

nombre_producto
Vacuum-Pro-Black        6203.0
Jacket-Winter-M         6185.0
Blender-XL-Red          6184.0
Sneakers-Urban-42       6085.0
Laptop-Gaming-16GB      4160.0
Tablet-Standard-64GB    4108.0
Phone-Pro-128GB         4088.0
Name: cantidad, dtype: float64


In [37]:
#Gasto en marketing por canal
marketing_canal = (

    marketing.groupby("canal")["gasto"]

    .sum()

    .sort_values(ascending=False)

)

print(marketing_canal)

canal
social         918043.21
organic        913533.01
paid_search    863088.21
Name: gasto, dtype: float64


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [38]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [39]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [40]:

# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT

    nombre_evento,

    COUNT(DISTINCT id_usuario) AS usuarios

FROM events

GROUP BY nombre_evento

ORDER BY usuarios DESC;
'''

totals = pd.read_sql(query_totals, con=engine)
totals


,nombre_evento,usuarios
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [41]:

# PARTE 2: Conversiones
# ======================

query_conversion = '''
SELECT

    nombre_evento,

    COUNT(DISTINCT id_usuario) AS usuarios

FROM events

GROUP BY nombre_evento
ORDER BY CASE nombre_evento

    WHEN 'first_visit' THEN 1

    WHEN 'select_item' THEN 2

    WHEN 'add_to_cart' THEN 3

    WHEN 'begin_checkout' THEN 4

    WHEN 'add_payment_info' THEN 5

    WHEN 'purchase' THEN 6

END;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion


,nombre_evento,usuarios
0,first_visit,7796
1,select_item,7582
2,add_to_cart,7634
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [42]:
# Calcular la tasa de conversión entre cada etapa

conversion["tasa_conversion_%"] = (

    conversion["usuarios"] /

    conversion["usuarios"].shift(1) * 100

).round(2)

# La primera etapa representa el 100%

conversion.loc[0, "tasa_conversion_%"] = 100

conversion


,nombre_evento,usuarios,tasa_conversion_%
0,first_visit,7796,100.00
1,select_item,7582,97.26
2,add_to_cart,7634,100.69
3,begin_checkout,7208,94.42
4,add_payment_info,6250,86.71
5,purchase,6240,99.84


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [43]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [50]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [51]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (

    SELECT

        id_usuario,

        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte

    FROM users

),

actividad AS (

    SELECT

        id_usuario,

        dias_despues_registro,

        activo

    FROM user_activity

    WHERE activo = 1

)

SELECT

    c.cohorte,

    COUNT(DISTINCT c.id_usuario) AS usuarios_iniciales,

    COUNT(DISTINCT CASE

        WHEN a.dias_despues_registro = 7

        THEN c.id_usuario

    END) AS retenido_w1,

    COUNT(DISTINCT CASE

        WHEN a.dias_despues_registro = 14

        THEN c.id_usuario

    END) AS retenido_w2,

    COUNT(DISTINCT CASE

        WHEN a.dias_despues_registro = 21

        THEN c.id_usuario

    END) AS retenido_w3,

    ROUND(

        COUNT(DISTINCT CASE

            WHEN a.dias_despues_registro = 7

            THEN c.id_usuario

        END)::numeric

        / COUNT(DISTINCT c.id_usuario) * 100,

        2

    ) AS semana_1,

    ROUND(

        COUNT(DISTINCT CASE

            WHEN a.dias_despues_registro = 14

            THEN c.id_usuario

        END)::numeric

        / COUNT(DISTINCT c.id_usuario) * 100,

        2

    ) AS semana_2,

    ROUND(

        COUNT(DISTINCT CASE

            WHEN a.dias_despues_registro = 21

            THEN c.id_usuario

        END)::numeric

        / COUNT(DISTINCT c.id_usuario) * 100,

        2

    ) AS semana_3

FROM cohortes c

LEFT JOIN actividad a

ON c.id_usuario = a.id_usuario

GROUP BY c.cohorte

ORDER BY c.cohorte;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01 00:00:00+00:00,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01 00:00:00+00:00,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01 00:00:00+00:00,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01 00:00:00+00:00,1687,695,676,706,41.20,40.07,41.85


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** ...
   - **H₁ (Hipótesis alternativa):** ...
   
**Test estadístico:** ...  
**Nivel de significancia alpha:** ...

In [3]:

import pandas as pd
experiment = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
experiment.head()


,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [4]:
#Analisis dataset
experiment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB


In [5]:
# totales de conversion por variante
experiment["variante"].value_counts()

tratamiento    5035
control        4965
Name: variante, dtype: int64

In [6]:
experiment["convirtio"].value_counts()

0    8401
1    1599
Name: convirtio, dtype: int64

In [9]:
#tasa de conversión.
experiment.groupby("variante")["convirtio"].mean()*100

variante
control        15.689829
tratamiento    16.285998
Name: convirtio, dtype: float64

In [8]:
#H0 hipotesis nula: La tasa de conversión del grupo control es igual a la del grupo tratamiento. Los cambios en la interfaz del checkout no tienen un impacto significativo.
#H1 hipotesis alternativa: La tasa de conversión del grupo tratamiento es diferente a la del grupo control. Los cambios en la interfaz sí tienen un impacto significativo.


In [4]:
#Test apropiado prueba Z, ya que la variable convirtio es binaria. 
from statsmodels.stats.proportion import proportions_ztest
# Número de conversiones por grupo

conversiones = experiment.groupby("variante")["convirtio"].sum()

# Número de usuarios por grupo

usuarios = experiment.groupby("variante")["convirtio"].count()

print(conversiones)

print(usuarios)


variante
control        779
tratamiento    820
Name: convirtio, dtype: int64
variante
control        4965
tratamiento    5035
Name: convirtio, dtype: int64


In [5]:
# Éxitos (conversiones)

count = conversiones.values

# Número de observaciones

nobs = usuarios.values

# Prueba Z para dos proporciones

z_stat, p_value = proportions_ztest(count, nobs)

print("Estadístico Z:", round(z_stat, 4))

print("Valor p:", round(p_value, 4))

Estadístico Z: -0.8133
Valor p: 0.4161


In [6]:
alpha = 0.05

if p_value < alpha:

    print("Se rechaza la hipótesis nula (H0).")

    print("Existe una diferencia significativa entre las tasas de conversión.")

else:

    print("No se rechaza la hipótesis nula (H0).")

    print("No existe evidencia suficiente para afirmar que el cambio en la UI tuvo un impacto significativo.")

No se rechaza la hipótesis nula (H0).
No existe evidencia suficiente para afirmar que el cambio en la UI tuvo un impacto significativo.


---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [ ]:
# (Pega aquí tu link)
# link de power bi o tableau
# link de one drive / google drive